In [2]:
# =====================================================================
# CELL 1 — Install Packages + Mount Drive
# =====================================================================
!pip install -U "transformers>=4.48.0" accelerate bitsandbytes qwen-vl-utils datasets tqdm -q

import os
from google.colab import drive
drive.mount('/content/drive')

STORAGE_ROOT = "/content/drive/MyDrive/Colab_Storage"
MODEL_CACHE_DIR = f"{STORAGE_ROOT}/model_cache"
RESULTS_ROOT = f"{STORAGE_ROOT}/gsv_math_results"

os.makedirs(MODEL_CACHE_DIR, exist_ok=True)
os.makedirs(RESULTS_ROOT, exist_ok=True)

os.environ["HF_HOME"] = MODEL_CACHE_DIR
os.environ["HF_DATASETS_CACHE"] = "/content/dataset_cache"

import torch
print(f"\nCUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.3/57.3 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.1/12.1 MB 111.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.0/41.0 MB 24.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 48.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.2/80.2 kB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 20.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 102.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.8/35.8 MB 19.2 MB/s eta 0:00:00
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

CUDA available: True
GPU: Tesla T4
VRAM: 15.6 GB


In [ ]:
# =====================================================================
# CELL 2 — Improved Parsing Logic v2 (tail-biased extraction)
# =====================================================================
import re

FINAL_ANSWER_PATTERNS = [
    r'\\boxed\{([^}]*)\}',
    r'[Ff]inal\s*[Aa]nswer\s*[:\-]?\s*(.{1,80})',
    r'[Tt]herefore[,\s]+(?:the\s+)?(?:answer|value|result)\s+is\s*[:\-]?\s*(.{1,80})',
    r'[Tt]he\s+answer\s+is\s*[:\-]?\s*(.{1,80})',
    r'[Ss]o\s+the\s+answer\s+is\s*[:\-]?\s*(.{1,80})',
    r'=\s*(\S+)\s*$',
]

def extract_final_answer_region(raw_text, tail_chars=300):
    for pattern in FINAL_ANSWER_PATTERNS:
        matches = list(re.finditer(pattern, raw_text, re.IGNORECASE | re.DOTALL))
        if matches:
            return matches[-1].group(1).strip()
    return raw_text[-tail_chars:] if len(raw_text) > tail_chars else raw_text

def clean_free_form(text):
    if not isinstance(text, str): return str(text)
    text = text.strip().lower()
    prefixes = ["the answer is", "therefore, the answer is", "so the answer is",
                "the value is", "answer is", "value is", "equals", "it is",
                "the final answer is", "final answer:", "answer:"]
    for prefix in prefixes:
        if text.startswith(prefix):
            text = text[len(prefix):].strip()
    match = re.match(r'^[a-zA-Z\s]+=\s*(.*)$', text)
    if match: text = match.group(1).strip()
    return text.rstrip('.!?*, ')

def get_most_similar(extraction, choices):
    distances = [-len(set(extraction.lower()) & set(choice.lower())) for choice in choices]
    return choices[distances.index(min(distances))]

def normalize_extracted_answer(extraction, choices, question_type, answer_type, precision=2):
    extraction = str(extraction).strip() if extraction else ""
    extraction = extract_final_answer_region(extraction)

    if question_type == 'multi_choice':
        letter = re.findall(r'\(([a-zA-Z])\)', extraction)
        extraction = letter[0].upper() if letter else extraction
        options = [chr(ord('A') + i) for i in range(len(choices))]
        if extraction in options:
            extraction = choices[options.index(extraction)]
        else:
            extraction = get_most_similar(clean_free_form(extraction), choices)
    else:
        cleaned = clean_free_form(extraction)
        if answer_type in ['integer', 'float']:
            numbers = re.findall(r'-?\d+\.?\d*', cleaned)
            extraction = numbers[-1] if numbers else cleaned
    return extraction

def is_correct(pred, gt, answer_type):
    if pred.lower().strip() == gt.lower().strip(): return 1
    if answer_type in ['integer', 'float']:
        try:
            if abs(float(pred) - float(gt)) < 1e-5: return 1
        except: pass
    return 0

In [3]:
# =====================================================================
# CELL 3 — Config
# =====================================================================
MODEL_ID = "Qwen/Qwen2.5-VL-7B-Instruct"
MAX_NEW_TOKENS = 512

RUN_NAME = "qwen25vl7b_zeroshot_progressive_res"
BATCH_SIZE = 50

RUN_DIR = f"{RESULTS_ROOT}/{RUN_NAME}"
import os
os.makedirs(RUN_DIR, exist_ok=True)

print(f"Target Model: {MODEL_ID}")
print(f"Run name:     {RUN_NAME}")
print(f"Checkpoints:  {RUN_DIR}")

Target Model: Qwen/Qwen2.5-VL-7B-Instruct
Run name:     qwen25vl7b_zeroshot_progressive_res
Checkpoints:  /content/drive/MyDrive/Colab_Storage/gsv_math_results/qwen25vl7b_zeroshot_progressive_res


In [4]:
# =====================================================================
# CELL 4 — Load Dataset Only (we already have model outputs saved!)
# =====================================================================
from datasets import load_dataset

print("Loading MathVista testmini dataset...")
mathvista = load_dataset("AI4Math/MathVista", split="testmini")
print(f"Loaded {len(mathvista)} test samples.")

# Build fast lookup
pid_lookup = {s["pid"]: s for s in mathvista}
print("Lookup table built.")

Loading MathVista testmini dataset...


Generating testmini split:   0%|          | 0/1000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/5141 [00:00<?, ? examples/s]

Loaded 1000 test samples.
Lookup table built.


In [ ]:
# =====================================================================
# CELL 5 — Re-score saved outputs with improved parser (NO GPU NEEDED)
# =====================================================================
import json, glob
from tqdm import tqdm

all_results = []
for f in sorted(glob.glob(f"{RUN_DIR}/batch_*.json")):
    with open(f) as fh:
        all_results.extend(json.load(fh))

print(f"Loaded {len(all_results)} saved results from {RUN_DIR}\n")

old_correct = sum(r["correct"] for r in all_results)

results = []
new_correct = 0

for r in tqdm(all_results, desc="Re-scoring with improved parser"):
    sample = pid_lookup.get(r["question_id"])
    if not sample:
        continue

    new_pred = normalize_extracted_answer(
        r["raw_answer"], sample.get("choices", []), sample["question_type"], sample["answer_type"]
    )
    new_flag = is_correct(new_pred, sample["answer"], sample["answer_type"])
    new_correct += new_flag

    results.append({
        "question_id": r["question_id"],
        "skills": r["skills"],
        "correct": new_flag,
        "raw_answer": r["raw_answer"],
        "parsed_answer": new_pred
    })

print(f"\nOld parser: {old_correct}/1000 = {old_correct/10:.1f}%")
print(f"New parser: {new_correct}/1000 = {new_correct/10:.1f}%")
print(f"Recovered:  +{new_correct - old_correct} samples")

Loaded 1000 saved results from /content/drive/MyDrive/Colab_Storage/gsv_math_results/qwen25vl7b_zeroshot_progressive_res



Re-scoring with improved parser: 100%|██████████| 1000/1000 [00:00<00:00, 16399.05it/s]


Old parser: 429/1000 = 42.9%
New parser: 619/1000 = 61.9%
Recovered:  +190 samples


In [ ]:
# =====================================================================
# CELL 6 — Save final re-scored results to Drive
# =====================================================================
import json

print(f"Total results: {len(results)} / {len(mathvista)}")

final_path = f"{RUN_DIR}/FINAL_results_v2_parser.json"
with open(final_path, "w") as fh:
    json.dump(results, fh)
print(f"Saved: {final_path}")

Total results: 1000 / 1000
Saved: /content/drive/MyDrive/Colab_Storage/gsv_math_results/qwen25vl7b_zeroshot_progressive_res/FINAL_results_v2_parser.json


In [ ]:
# =====================================================================
# CELL 7 — Calculate and Print Metrics
# =====================================================================
skill_to_category = {
    "geometry reasoning": "geometry",
    "arithmetic reasoning": "arithmetic",
    "algebraic reasoning": "algebra",
    "logical reasoning": "logic",
    "numeric commonsense": "numeric",
    "scientific reasoning": "scientific",
    "statistical reasoning": "statistical",
}

categories = ["all", "geometry", "arithmetic", "algebra", "logic", "numeric", "scientific", "statistical"]
metrics = {cat: {"correct": 0, "total": 0} for cat in categories}

for res in results:
    correct = res["correct"]
    metrics["all"]["correct"] += correct
    metrics["all"]["total"] += 1

    for skill in res.get("skills", []):
        cat = skill_to_category.get(skill)
        if cat in metrics:
            metrics[cat]["correct"] += correct
            metrics[cat]["total"] += 1

print("\n" + "="*60)
print(f"{'Qwen2.5-VL-7B — Zero-Shot (v2 Parser)':^60}")
print("="*60)
print(f"{'Category':<20} | {'Correct':<10} | {'Total':<10} | {'Accuracy':<10}")
print("-"*60)

for cat in categories:
    correct = metrics[cat]["correct"]
    total = metrics[cat]["total"]
    acc = (correct / total * 100) if total > 0 else 0.0
    print(f"{cat.capitalize():<20} | {correct:<10} | {total:<10} | {acc:.2f}%")

print("="*60)
print(f"\nResults saved at: {RUN_DIR}")


           Qwen2.5-VL-7B — Zero-Shot (v2 Parser)            
Category             | Correct    | Total      | Accuracy  
------------------------------------------------------------
All                  | 619        | 1000       | 61.90%
Geometry             | 136        | 239        | 56.90%
Arithmetic           | 204        | 353        | 57.79%
Algebra              | 161        | 281        | 57.30%
Logic                | 8          | 37         | 21.62%
Numeric              | 61         | 144        | 42.36%
Scientific           | 65         | 122        | 53.28%
Statistical          | 237        | 301        | 78.74%

Results saved at: /content/drive/MyDrive/Colab_Storage/gsv_math_results/qwen25vl7b_zeroshot_progressive_res


In [5]:
# =====================================================================
# EXTRA CELL: LOAD QWEN 2.5 MODEL + PROCESSOR
# =====================================================================
from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor, BitsAndBytesConfig
import torch

MODEL_ID = "Qwen/Qwen2.5-VL-7B-Instruct"

print(f"Loading {MODEL_ID} in 4-bit...")
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
)

model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    MODEL_ID,
    quantization_config=quantization_config,
    device_map="auto"
)
processor = AutoProcessor.from_pretrained(MODEL_ID)

print(" Model and Processor loaded successfully! You can now run the VDS cell.")

Loading Qwen/Qwen2.5-VL-7B-Instruct in 4-bit...


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/729 [00:00<?, ?it/s]

 Model and Processor loaded successfully! You can now run the VDS cell.


In [6]:
import os, json, time, re
from tqdm.auto import tqdm
from PIL import Image
from qwen_vl_utils import process_vision_info
from datasets import load_dataset
from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor, BitsAndBytesConfig
import torch

# =====================================================================
# 1. LOAD THE QWEN 2.5 MODEL FIRST (Fixes NameError!)
# =====================================================================
MODEL_ID = "Qwen/Qwen2.5-VL-7B-Instruct"
print(f"Loading {MODEL_ID} in 4-bit...")

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
)

model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    MODEL_ID,
    quantization_config=quantization_config,
    device_map="auto"
)
processor = AutoProcessor.from_pretrained(MODEL_ID)
print("Model loaded successfully!")

# =====================================================================
# 2. DEFINE THE SMART PARSER
# =====================================================================
FINAL_ANSWER_PATTERNS = [
    r'\\boxed\{([^}]*)\}',
    r'[Ff]inal\s*[Aa]nswer\s*[:\-]?\s*(.{1,80})',
    r'[Tt]herefore[,\\s]+(?:the\s+)?(?:answer|value|result)\s+is\s*[:\-]?\s*(.{1,80})',
    r'[Tt]he\s+answer\s+is\s*[:\-]?\s*(.{1,80})',
    r'[Ss]o\s+the\s+answer\s+is\s*[:\-]?\s*(.{1,80})',
    r'=\s*(\S+)\s*$',
]

def extract_final_answer_region(raw_text, tail_chars=300):
    for pattern in FINAL_ANSWER_PATTERNS:
        matches = list(re.finditer(pattern, raw_text, re.IGNORECASE | re.DOTALL))
        if matches: return matches[-1].group(1).strip()
    return raw_text[-tail_chars:] if len(raw_text) > tail_chars else raw_text

def clean_free_form(text):
    if not isinstance(text, str): return str(text)
    text = text.strip().lower()
    for prefix in ["the answer is", "therefore, the answer is", "so the answer is", "the value is", "answer is", "value is", "equals", "it is", "the final answer is", "final answer:", "answer:"]:
        if text.startswith(prefix):
            text = text[len(prefix):].strip()
    match = re.match(r'^[a-zA-Z\s]+=\s*(.*)$', text)
    if match: text = match.group(1).strip()
    return text.rstrip('.!?*, ')

def get_most_similar(extraction, choices):
    if not choices: return extraction
    distances = [-len(set(extraction.lower()) & set(choice.lower())) for choice in choices]
    return choices[distances.index(min(distances))]

def normalize_extracted_answer(extraction, choices, question_type, answer_type):
    extraction = str(extraction).strip() if extraction else ""
    extraction = extract_final_answer_region(extraction)

    if question_type == 'multi_choice':
        letter = re.findall(r'\(([a-zA-Z])\)', extraction)
        extraction = letter[0].upper() if letter else extraction
        options = [chr(ord('A') + i) for i in range(len(choices))]
        if extraction in options:
            extraction = choices[options.index(extraction)]
        else:
            extraction = get_most_similar(clean_free_form(extraction), choices)
    else:
        cleaned = clean_free_form(extraction)
        if answer_type in ['integer', 'float']:
            numbers = re.findall(r'-?\d+\.?\d*', cleaned)
            extraction = numbers[-1] if numbers else cleaned
        else:
            extraction = cleaned
    return extraction

def is_correct(pred, gt, answer_type):
    if str(pred).lower().strip() == str(gt).lower().strip(): return 1
    if answer_type in ['integer', 'float']:
        try:
            if abs(float(pred) - float(gt)) < 1e-5: return 1
        except: pass
    return 0

# =====================================================================
# 3. RUN THE VDS BLIND ABLATION
# =====================================================================
print("Loading MathVista testmini for VDS Ablation...")
eval_dataset = load_dataset("AI4Math/MathVista", split="testmini")

print("Starting Blind Ablation (VDS) pass for Qwen 2.5 Zero-Shot...")
blank_image = Image.new('RGB', (224, 224), color='black')

# Corrected the file name to Qwen 2.5!
VDS_RESULTS_FILE = "/content/drive/MyDrive/Colab_Storage/gsv_math_results/qwen2.5_zeroshot_BLIND.json"
vds_results = []
if os.path.exists(VDS_RESULTS_FILE):
    with open(VDS_RESULTS_FILE, "r") as f:
        vds_results = json.load(f)
completed_pids = {res["pid"] for res in vds_results}

remaining_samples = [s for s in eval_dataset if s["pid"] not in completed_pids]

for sample in tqdm(remaining_samples):
    pid = sample["pid"]
    question = sample["query"]
    gt = sample["answer"]

    # Pass the BLANK image
    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": blank_image},
                {"type": "text", "text": question}
            ]
        }
    ]

    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    image_inputs, video_inputs = process_vision_info(messages)

    inputs = processor(
        text=[text], images=image_inputs, videos=video_inputs, padding=True, return_tensors="pt"
    ).to(model.device)

    with torch.no_grad():
        generated_ids = model.generate(**inputs, max_new_tokens=1024)
        generated_ids_trimmed = [out_ids[len(in_ids):] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)]
        output_text = processor.batch_decode(generated_ids_trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False)[0]

    parsed_ans = normalize_extracted_answer(output_text, sample.get("choices", []), sample["question_type"], sample["answer_type"])
    correct = is_correct(parsed_ans, gt, sample["answer_type"])

    vds_results.append({
        "pid": pid, "question": question, "raw_response": output_text,
        "parsed_response": parsed_ans, "ground_truth": gt, "correct": correct
    })

    if len(vds_results) % 10 == 0:
        with open(VDS_RESULTS_FILE, "w") as f:
            json.dump(vds_results, f, indent=4)
        time.sleep(1) # Give Drive time to sync

with open(VDS_RESULTS_FILE, "w") as f:
    json.dump(vds_results, f, indent=4)

print("="*50)
print(f"BLIND ACCURACY (Qwen 2.5): {(sum(r['correct'] for r in vds_results) / len(vds_results)) * 100:.2f}%")
print("="*50)

Loading Qwen/Qwen2.5-VL-7B-Instruct in 4-bit...


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/729 [00:00<?, ?it/s]

Model loaded successfully!
Loading MathVista testmini for VDS Ablation...
Starting Blind Ablation (VDS) pass for Qwen 2.5 Zero-Shot...


  0%|          | 0/20 [00:00<?, ?it/s]

/usr/local/lib/python3.13/dist-packages/bitsandbytes/backends/cuda/ops.py:957: UserWarning: inner dimension (3420) is not aligned for fast kernel with blocksize=64, falling back to slower implementation.
  warn(


BLIND ACCURACY (Qwen 2.5): 31.00%
